# Evaluating the LangGraph + Qdrant Hybrid RAG Pipeline with DeepEval

A companion notebook to `LangGraph_Qdrant_Hybrid_Rerank.ipynb`, focused entirely on **evaluation** using the open-source [DeepEval](https://github.com/confident-ai/deepeval) library.

Covers:
- DeepEval fundamentals: `LLMTestCase`, metrics, `evaluate()`
- Rebuilding a minimal version of the RAG pipeline to generate real outputs to evaluate
- Building a small evaluation dataset (queries + ground-truth context + reference answers)
- Core **RAG metrics**: Faithfulness, Answer Relevancy, Contextual Precision, Contextual Recall, Contextual Relevancy
- **Hallucination** metric
- **LLM-as-a-judge** via `GEval` - custom, criteria-based scoring
- Aggregating results into a readable scorecard
- Notes on CI integration and regression testing

**Environment:** Google Colab
**Judge / generation LLM:** OpenAI (via API key stored in Colab Secrets) - DeepEval uses an LLM internally to score most of these metrics


---
## 1. Why RAG Evaluation Needs Its Own Toolkit

A RAG pipeline has (at least) two moving parts that can each fail independently:

1. **Retrieval** - did we fetch the right context for the query?
2. **Generation** - given that context, did the LLM produce an answer that's faithful to it and actually answers the question?

Generic "is this a good answer" scoring can't tell you *which* part broke. If the answer is wrong, was it because retrieval missed the right document, or because the LLM hallucinated despite having the right context?

**DeepEval** is an open-source evaluation framework purpose-built for LLM/RAG pipelines. It provides:
- Ready-made metrics for retrieval quality (precision/recall/relevancy of retrieved context)
- Ready-made metrics for generation quality (faithfulness to context, answer relevancy, hallucination)
- A generic **LLM-as-a-judge** metric (`GEval`) for custom, criteria-based scoring when the built-in metrics don't cover your use case
- A `pytest`-style workflow (`deepeval test run`) so RAG evaluation can live in CI, not just notebooks

### 1.1 Core DeepEval Concepts

| Concept | Description |
|---|---|
| `LLMTestCase` | A single evaluation unit: `input` (query), `actual_output` (generated answer), `retrieval_context` (chunks the RAG system retrieved), and optionally `expected_output` (reference/ground-truth answer) |
| `Metric` | A scoring function (e.g., `FaithfulnessMetric`) that takes a test case and produces a score (usually 0-1) plus a pass/fail against a threshold, and a reason |
| `evaluate()` | Runs a list of test cases against a list of metrics and prints/returns a results table |
| `GEval` | A generic metric where **you** define the evaluation criteria in plain English, and an LLM judges the output against those criteria - this is the "LLM-as-a-judge" pattern |

### 1.2 Retrieval Metrics vs Generation Metrics

DeepEval's RAG metrics split cleanly along the two failure points above:

**Retrieval-focused** (do we have the right context?)
- `ContextualPrecisionMetric` - of the retrieved chunks, how many are actually relevant, and are relevant ones ranked highly?
- `ContextualRecallMetric` - did retrieval capture everything needed to produce the expected answer?
- `ContextualRelevancyMetric` - how relevant is the retrieved context to the query overall?

**Generation-focused** (given the context, is the answer good?)
- `FaithfulnessMetric` - does the answer avoid claims that contradict or aren't supported by the retrieved context? (this is DeepEval's hallucination-vs-context check)
- `AnswerRelevancyMetric` - does the answer actually address the query (regardless of correctness against context)?
- `HallucinationMetric` - a related but distinct check: does the output contradict a provided ground-truth "context" document, useful for factuality checks outside pure RAG-context grounding

We'll compute all of these, plus a custom `GEval` judge for a criterion DeepEval doesn't ship out of the box (e.g., "is the answer appropriately concise and instructional in tone").


---
## 2. Setup & Environment

Installs:
- `deepeval` - the evaluation framework
- `langgraph`, `langchain`, `langchain-openai` - to rebuild the RAG pipeline
- `qdrant-client`, `fastembed`, `sentence-transformers` - retrieval + rerank stack (same as the main notebook)

### API Key via Google Colab Secrets

Same as the main notebook: store your key under the 🔑 **Secrets** panel as `OPENAI_API_KEY` with notebook access enabled. DeepEval uses this key to run its default GPT-4o-mini judge model for most metrics (you can swap the judge model - see section 8).


### 📦 Cell 0 — Install the dependencies

Installs `deepeval` (the evaluation framework this whole notebook is about) alongside the same LangGraph/Qdrant/embedding stack from the main notebook — needed because Section 3 below rebuilds a small copy of that RAG pipeline just to have real outputs to evaluate.

In [1]:
!pip install -q deepeval langgraph langchain langchain-openai langchain-core \
    qdrant-client fastembed sentence-transformers

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 🔑 Cell 1 — Load the OpenAI key from Colab Secrets

Same pattern as the main notebook: pulls the key from Colab's Secrets panel instead of hardcoding it. This key does double duty here — it powers the rebuilt pipeline's `generate` step, and it's also what DeepEval uses by default to run its own "judge" LLM (`gpt-4o-mini`) that scores every metric below.

In [2]:
# --- OpenAI API key from local environment variable ---
import os

# Reads the key directly from the environment. Set it beforehand, e.g.:
#   Windows (PowerShell): $env:OPENAI_API_KEY = "sk-..."
#   macOS/Linux:           export OPENAI_API_KEY="sk-..."
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found in environment - set it before running this notebook."
print("OpenAI API key found in environment ✔")


OpenAI API key found in environment ✔


### 🧰 Cell 2 — Import everything: the RAG stack, plus DeepEval's evaluation pieces

Two groups of imports: the familiar retrieval/generation stack (`QdrantClient`, `TextEmbedding`, `CrossEncoder`, `StateGraph`) to rebuild the pipeline, and DeepEval's own building blocks — `LLMTestCase` (one thing-to-grade), and a set of ready-made metrics (`FaithfulnessMetric`, `AnswerRelevancyMetric`, etc.) plus `GEval` for custom criteria. Think of `LLMTestCase` as one row on an exam, and each metric as one grading rubric applied to that row.

In [3]:
# Type hints used for defining state schemas and improving code readability.
# - TypedDict: Creates strongly-typed dictionary structures.
# - List: Represents a list of objects.
from typing import TypedDict, List

# Qdrant client for interacting with the vector database.
# - QdrantClient: Connects to the Qdrant server.
# - qmodels: Contains helper models for collections, search parameters, etc.
from qdrant_client import QdrantClient, models as qmodels

# FastEmbed provides lightweight embedding models.
# - TextEmbedding: Generates dense vector embeddings.
# - SparseTextEmbedding: Generates sparse embeddings for hybrid search.
from fastembed import TextEmbedding, SparseTextEmbedding

# CrossEncoder model used for reranking retrieved documents.
# It computes relevance scores between the query and each retrieved document.
from sentence_transformers import CrossEncoder

# ChatOpenAI provides access to OpenAI chat models through LangChain.
from langchain_openai import ChatOpenAI

# LangGraph components used to build stateful AI workflows.
# - StateGraph: Creates a graph-based workflow.
# - START: Entry point of the graph.
# - END: Exit point of the graph.
from langgraph.graph import StateGraph, START, END

# DeepEval is an evaluation framework for testing the quality
# of LLM applications and Retrieval-Augmented Generation (RAG) systems.
from deepeval import evaluate

# LLMTestCase defines the inputs, outputs, and retrieval context
# used during evaluation.
from deepeval.test_case import LLMTestCase

# Pre-built evaluation metrics available in DeepEval.
#
# - FaithfulnessMetric:
#     Checks whether the generated answer is grounded in the retrieved context.
#
# - AnswerRelevancyMetric:
#     Measures how well the answer addresses the user's question.
#
# - ContextualPrecisionMetric:
#     Evaluates whether retrieved documents are relevant and free from
#     unnecessary information.
#
# - ContextualRecallMetric:
#     Measures whether the retrieval process captured all relevant information.
#
# - ContextualRelevancyMetric:
#     Evaluates the overall usefulness and relevance of the retrieved context.
#
# - HallucinationMetric:
#     Detects whether the model generates unsupported or fabricated information.
#
# - GEval:
#     A customizable LLM-as-a-Judge evaluation framework that allows
#     user-defined evaluation criteria.
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    HallucinationMetric,
    GEval,
)

# Defines which fields from an LLM test case should be evaluated
# when creating custom GEval metrics.
from deepeval.test_case import LLMTestCaseParams

C:\Users\Avado\AppData\Local\Temp\ipykernel_45560\739480251.py:73: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


---
## 3. Rebuilding a Minimal RAG Pipeline

To evaluate a RAG pipeline, we need real outputs from it - not hand-written examples. This section rebuilds a **condensed version** of the LangGraph + Qdrant + hybrid search + rerank pipeline from the main notebook (same corpus, same models) so this notebook is self-contained. If you already have `graph` compiled from the main notebook in the same runtime, you can skip straight to Section 4 and reuse it.


### 🔤 Cell 3 — Load the same dense + sparse embedding models as the main pipeline

Sets up `BAAI/bge-small-en-v1.5` (dense, meaning-based) and `Qdrant/bm25` (sparse, keyword-based) — this is the hybrid search foundation from *Retrieval Optimization Techniques*: dense catches semantic matches, sparse catches exact terms neither alone reliably finds.

In [4]:
# Initialize the dense embedding model.
#
# This model converts text into dense numerical vectors that capture
# semantic meaning, making it suitable for similarity search.
#
# Model:
#   BAAI/bge-small-en-v1.5
# Framework:
#   FastEmbed (ONNX-based, runs locally)
dense_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# Initialize the sparse embedding model.
#
# Sparse embeddings capture keyword importance similar to BM25,
# making them useful for lexical retrieval.
#
# This model will be combined with dense embeddings to perform
# Hybrid Search.
sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25"
)

# Dimension of embeddings produced by the BGE Small model.
DENSE_DIM = 384

# EMBEDDING HELPER FUNCTIONS

def embed_dense(texts):
    """
    Generate dense embeddings for a list of text documents.

    Parameters:
        texts (List[str]):
            Input text documents.

    Returns:
        List[List[float]]:
            Dense embedding vectors represented as Python lists.
    """
    return [vector.tolist() for vector in dense_model.embed(texts)]


def embed_sparse(texts):
    """
    Generate sparse embeddings for a list of text documents.

    Parameters:
        texts (List[str]):
            Input text documents.

    Returns:
        List:
            Sparse embedding objects containing:
            - indices
            - values
    """
    return list(sparse_model.embed(texts))

# LOAD CROSS-ENCODER RERANKER

# Initialize a CrossEncoder model.
#
# Unlike embedding models, the CrossEncoder jointly processes
# both the query and document to predict a highly accurate
# semantic relevance score.
#
# It is typically used after retrieval to rerank the
# top candidate documents.
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

# INITIALIZE THE LANGUAGE MODEL

# Create the LLM used for answer generation.
#
# Temperature = 0 makes responses deterministic,
# which is generally preferred for RAG applications.
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


# Confirm that all models have been initialized successfully.
print("Models loaded ✔")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Models loaded ✔


### 🗄️ Cell 4 — Spin up a fresh, in-memory Qdrant collection

Creates a throwaway `:memory:` Qdrant instance (nothing persists after the notebook session ends) with two named vectors — `dense` and `sparse` — per point, exactly like the main pipeline's collection. This is a self-contained copy just for generating fresh evaluation data, not the same collection the main notebook writes to.

In [5]:
# Name of the Qdrant collection that will store
# documents and their corresponding embeddings.
COLLECTION_NAME = "langgraph_hybrid_eval_demo"

# Create an in-memory Qdrant instance.
#
# Using ":memory:" keeps the database in RAM, making it
# ideal for notebooks, tutorials, and experimentation.
#
# Note:
# The collection will be deleted once the Python session ends.
client = QdrantClient(":memory:")

# Create a collection capable of storing both
# dense and sparse embeddings for Hybrid Search.
client.create_collection(
    collection_name=COLLECTION_NAME,

    # Configuration for dense vector embeddings.
    vectors_config={
        "dense": qmodels.VectorParams(
            size=DENSE_DIM,                     # Dimension of dense embeddings
            distance=qmodels.Distance.COSINE    # Similarity metric for vector search
        )
    },

    # Configuration for sparse vectors used for
    # keyword-based retrieval.
    sparse_vectors_config={
        "sparse": qmodels.SparseVectorParams()
    },
)

# Confirm successful collection creation.
print(f"Collection '{COLLECTION_NAME}' created ✔")

documents = [
    {"id": 0, "text": "LangGraph is a library for building stateful, multi-actor applications with LLMs, built on top of LangChain.", "topic": "langgraph"},
    {"id": 1, "text": "A StateGraph in LangGraph defines nodes, edges, and a shared state object that flows through the graph.", "topic": "langgraph"},
    {"id": 2, "text": "Unlike LCEL chains, which are directed acyclic graphs, LangGraph supports cycles and conditional branching.", "topic": "langgraph"},
    {"id": 3, "text": "Qdrant is an open-source vector database written in Rust, optimized for high-performance similarity search.", "topic": "qdrant"},
    {"id": 4, "text": "Qdrant supports named vectors, allowing multiple embeddings (e.g., dense and sparse) to be stored per point.", "topic": "qdrant"},
    {"id": 5, "text": "Hybrid search combines dense semantic vectors with sparse lexical vectors to improve retrieval accuracy.", "topic": "hybrid_search"},
    {"id": 6, "text": "Reciprocal Rank Fusion (RRF) is a common method for merging ranked lists from multiple retrieval systems.", "topic": "hybrid_search"},
    {"id": 7, "text": "BM25 is a classic sparse retrieval algorithm based on term frequency and inverse document frequency.", "topic": "hybrid_search"},
    {"id": 8, "text": "Dense embeddings encode semantic meaning, allowing retrieval of conceptually similar text even without shared keywords.", "topic": "embeddings"},
    {"id": 9, "text": "Cross-encoder rerankers score a query-document pair jointly, producing more accurate relevance scores than bi-encoders.", "topic": "reranking"},
    {"id": 10, "text": "BAAI/bge-reranker-base is an open-source cross-encoder model commonly used for reranking retrieved passages.", "topic": "reranking"},
    {"id": 11, "text": "Reranking is typically applied to a small candidate set (e.g., top 20-50) after initial retrieval, since cross-encoders are more compute-intensive.", "topic": "reranking"},
    {"id": 12, "text": "Agentic RAG systems combine retrieval-augmented generation with agent-style decision making, such as deciding whether to retrieve at all.", "topic": "agents"},
    {"id": 13, "text": "LangGraph's checkpointing feature allows persisting graph state, enabling resumability and human-in-the-loop workflows.", "topic": "langgraph"},
    {"id": 14, "text": "Conditional edges in LangGraph route execution to different nodes based on the current state, enabling dynamic control flow.", "topic": "langgraph"},
    {"id": 15, "text": "Open-source embedding models like bge-small-en-v1.5 can run efficiently on CPU, making them practical for local development.", "topic": "embeddings"},
    {"id": 16, "text": "Vector databases like Qdrant, Weaviate, and Milvus are optimized for approximate nearest neighbor search at scale.", "topic": "qdrant"},
    {"id": 17, "text": "SPLADE is a sparse retrieval model that learns term expansions, unlike BM25 which relies purely on exact term matches.", "topic": "hybrid_search"},
]

# Extract the text content from each document.
# These texts will be converted into dense and sparse embeddings.
texts = [doc["text"] for doc in documents]

# Generate dense embeddings that capture the semantic
# meaning of each document.
dense_vectors = embed_dense(texts)

# Generate sparse embeddings that capture keyword importance
# for lexical retrieval.
sparse_vectors = embed_sparse(texts)

# Create Qdrant Point objects.
#
# Each point contains:
# - A unique document ID
# - Dense embedding
# - Sparse embedding
# - Metadata (payload)
points = [
    qmodels.PointStruct(
        id=doc["id"],

        # Store both dense and sparse vectors
        # for Hybrid Search.
        vector={
            "dense": dense_vector,

            "sparse": qmodels.SparseVector(
                indices=sparse_vector.indices.tolist(),
                values=sparse_vector.values.tolist(),
            ),
        },

        # Additional metadata stored alongside
        # each document.
        payload={
            "text": doc["text"],
            "topic": doc["topic"],
        },
    )

    for doc, dense_vector, sparse_vector in zip(
        documents,
        dense_vectors,
        sparse_vectors,
    )
]

# Insert all documents into the Qdrant collection.
#
# Upsert will:
# - Insert new documents.
# - Update existing documents if the IDs already exist.
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

# Display the number of successfully stored documents.
print(f"Upserted {len(points)} points.")

Collection 'langgraph_hybrid_eval_demo' created ✔


Upserted 18 points.


### 🕸️ Cell 5 — Rebuild the whole retrieve → rerank → generate pipeline in one cell

This condenses everything from the main notebook into one place: `hybrid_search()` runs dense + sparse search and fuses them with **RRF** (positional-only fusion, no reading involved — see the *Advanced RAG Architectures* notes); `RAGState` is the shared "clipboard" LangGraph nodes read from and write to; `retrieve_node` calls hybrid search, `rerank_node` re-scores the candidates with a cross-encoder (reads query+doc *together*, unlike the bi-encoder used for the first pass), and `generate_node` builds the final prompt and calls the LLM. The last few lines wire these three nodes into a compiled `graph` — the exact same shape the main notebook builds, just recreated here so this notebook can run standalone.

In [6]:
def hybrid_search(
    query: str,
    top_k: int = 10,
    prefetch_k: int = 20,
):
    """
    Perform Hybrid Search using Reciprocal Rank Fusion (RRF).

    The retrieval process combines:
        1. Dense Search (semantic similarity)
        2. Sparse Search (keyword matching)

    Qdrant first retrieves candidate documents independently
    from both indexes and then merges the rankings using
    Reciprocal Rank Fusion (RRF).

    Parameters:
        query (str):
            User's search query.

        top_k (int):
            Number of final search results.

        prefetch_k (int):
            Number of candidate documents retrieved from
            each retrieval strategy before fusion.

    Returns:
        List[ScoredPoint]:
            Ranked search results.
    """

    # Generate dense and sparse query embeddings.
    q_vec = embed_dense([query])[0]
    q_sparse = embed_sparse([query])[0]

    # Execute Hybrid Search.
    results = client.query_points(
        collection_name=COLLECTION_NAME,

        prefetch=[
            # Dense semantic retrieval
            qmodels.Prefetch(
                query=q_vec,
                using="dense",
                limit=prefetch_k,
            ),

            # Sparse keyword retrieval
            qmodels.Prefetch(
                query=qmodels.SparseVector(
                    indices=q_sparse.indices.tolist(),
                    values=q_sparse.values.tolist(),
                ),
                using="sparse",
                limit=prefetch_k,
            ),
        ],

        # Merge both rankings using RRF.
        query=qmodels.FusionQuery(
            fusion=qmodels.Fusion.RRF
        ),

        # Return the final top-k results.
        limit=top_k,
    )

    return results.points

# DEFINE LANGGRAPH STATE

class RAGState(TypedDict):
    """
    Shared state passed between all nodes in the LangGraph workflow.
    """

    # User's question.
    query: str

    # Documents retrieved from Hybrid Search.
    retrieved_docs: List[dict]

    # Top-ranked documents after CrossEncoder reranking.
    reranked_docs: List[dict]

    # Final LLM-generated answer.
    answer: str

# NODE 1: RETRIEVE DOCUMENTS

def retrieve_node(state: RAGState) -> dict:
    """
    Retrieve relevant documents using Hybrid Search.
    """

    results = hybrid_search(
        state["query"],
        top_k=10,
    )

    docs = [
        {
            "text": result.payload["text"],
            "topic": result.payload["topic"],
            "score": result.score,
        }
        for result in results
    ]

    return {
        "retrieved_docs": docs
    }

# NODE 2: RERANK DOCUMENTS

def rerank_node(state: RAGState) -> dict:
    """
    Improve document ranking using a CrossEncoder model.
    """

    # Create query-document pairs.
    pairs = [
        (state["query"], doc["text"])
        for doc in state["retrieved_docs"]
    ]

    # Compute semantic relevance scores.
    scores = reranker.predict(pairs)

    # Sort by descending relevance.
    combined = sorted(
        zip(state["retrieved_docs"], scores),
        key=lambda item: item[1],
        reverse=True,
    )

    # Keep the three most relevant documents.
    top_docs = [
        {
            **doc,
            "rerank_score": float(score),
        }
        for doc, score in combined[:3]
    ]

    return {
        "reranked_docs": top_docs
    }

# NODE 3: GENERATE ANSWER

def generate_node(state: RAGState) -> dict:
    """
    Generate the final answer using the reranked documents
    as retrieval context.
    """

    # Build the retrieval context.
    context = "\n".join(
        f"- {doc['text']}"
        for doc in state["reranked_docs"]
    )

    # Construct the RAG prompt.
    prompt = (
        "Answer the question using only the context below. "
        "If the context is insufficient, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {state['query']}\n\n"
        "Answer:"
    )

    # Generate the response.
    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

# BUILD THE LANGGRAPH WORKFLOW

# Create a graph using the shared workflow state.
graph_builder = StateGraph(RAGState)

# Register workflow nodes.
graph_builder.add_node("retrieve", retrieve_node)
graph_builder.add_node("rerank", rerank_node)
graph_builder.add_node("generate", generate_node)

# Define execution order.
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "rerank")
graph_builder.add_edge("rerank", "generate")
graph_builder.add_edge("generate", END)

# Compile the graph into an executable workflow.
graph = graph_builder.compile()

print("RAG pipeline rebuilt and compiled ✔")

RAG pipeline rebuilt and compiled ✔


---
## 4. Building an Evaluation Dataset

DeepEval's RAG metrics need, per example:
- `input` - the query
- `actual_output` - what the pipeline generated
- `retrieval_context` - the chunks the pipeline actually retrieved (used by faithfulness/relevancy metrics)
- `expected_output` - a reference/"gold" answer (needed for `ContextualRecallMetric` and useful for `GEval`)

We define a small hand-curated eval set: queries paired with reference answers. We'll run the pipeline live to get `actual_output` and `retrieval_context` for each.


### 📋 Cell 6 — Hand-write the evaluation set: questions + a reference answer

This is the human-verified **ground truth** from earlier in this session's discussion — six questions, each with an `expected_output` a person wrote and confirmed as correct. Nothing here is generated by the pipeline yet; this is the *answer key* the pipeline's real outputs get checked against.

In [7]:
eval_dataset = [
    {
        "query": "What is the difference between LangGraph and a LangChain LCEL chain?",
        "expected_output": "LCEL chains are linear, acyclic pipelines, while LangGraph supports stateful graphs with cycles and conditional branching, making it suited for agentic workflows.",
    },
    {
        "query": "How does hybrid search combine dense and sparse retrieval?",
        "expected_output": "Hybrid search runs dense (semantic) and sparse (lexical/BM25-style) retrieval separately, then merges the two ranked lists using a fusion method such as Reciprocal Rank Fusion (RRF).",
    },
    {
        "query": "What open-source model can be used for reranking retrieved documents?",
        "expected_output": "Open-source cross-encoder models like cross-encoder/ms-marco-MiniLM-L-6-v2 or BAAI/bge-reranker-base can be used to rerank retrieved documents.",
    },
    {
        "query": "Why would I use RRF fusion instead of dense search alone?",
        "expected_output": "RRF fusion combines dense and sparse retrieval results, capturing both semantic similarity and exact keyword matches, which can outperform dense-only search especially for queries with specific terms.",
    },
    {
        "query": "What is Qdrant and what makes it different from a regular database?",
        "expected_output": "Qdrant is an open-source vector database optimized for high-performance similarity search over embeddings, unlike regular databases which are optimized for exact-match or relational queries.",
    },
    {
        "query": "What programming language is Qdrant written in?",
        "expected_output": "Qdrant is written in Rust.",
    },
]

print(f"Eval dataset size: {len(eval_dataset)} examples")

Eval dataset size: 6 examples


### Running the Pipeline to Collect Outputs

For each query we run the compiled LangGraph pipeline and capture the reranked context plus the generated answer - these become the `retrieval_context` and `actual_output` for each `LLMTestCase`.


### ▶️ Cell 7 — Run the real pipeline on every question, and save what it actually produced

For each of the 6 questions, this calls `graph.invoke()` — running the real retrieve→rerank→generate pipeline — and saves the `actual_output` (what the model said) and the `retrieval_context` (the reranked chunks it was actually given). This is the critical step that makes evaluation possible: DeepEval needs *real* outputs, not hypothetical ones, to judge faithfulness and relevancy against.

In [8]:
# Store the outputs generated for each evaluation example.
#
# Each entry will contain:
# - User query
# - Expected (reference) answer
# - Actual answer generated by the RAG pipeline
# - Retrieval context used to generate the answer
results = []

# Execute the RAG pipeline for every evaluation example.
for example in eval_dataset:

    # Run the complete LangGraph workflow.
    output = graph.invoke(
        {
            "query": example["query"]
        }
    )

    # Save the information required for DeepEval metrics.
    results.append(
        {
            "query": example["query"],

            # Ground-truth/reference answer.
            "expected_output": example["expected_output"],

            # Answer generated by the RAG pipeline.
            "actual_output": output["answer"],

            # Context actually used by the LLM after reranking.
            "retrieval_context": [
                document["text"]
                for document in output["reranked_docs"]
            ],
        }
    )

# REVIEW GENERATED RESULTS

# Display the generated answer and retrieval context
# for each evaluation example before running DeepEval.
for result in results:

    print(f"\nQ: {result['query']}")
    print(f"A: {result['actual_output']}")

    # Display the number of retrieved context chunks
    # supplied to the language model.
    print(
        f"Context used: {len(result['retrieval_context'])} chunks"
    )


Q: What is the difference between LangGraph and a LangChain LCEL chain?
A: The difference between LangGraph and a LangChain LCEL chain is that LangGraph supports cycles and conditional branching, while LCEL chains are directed acyclic graphs. Additionally, LangGraph is designed for building stateful, multi-actor applications with LLMs and includes a StateGraph that defines nodes, edges, and a shared state object.
Context used: 3 chunks

Q: How does hybrid search combine dense and sparse retrieval?
A: Hybrid search combines dense semantic vectors, which capture the meaning of the content, with sparse lexical vectors, which focus on specific terms and their occurrences, to improve retrieval accuracy.
Context used: 3 chunks

Q: What open-source model can be used for reranking retrieved documents?
A: BAAI/bge-reranker-base is an open-source model that can be used for reranking retrieved documents.
Context used: 3 chunks

Q: Why would I use RRF fusion instead of dense search alone?
A: You 

---
## 5. Constructing DeepEval Test Cases

Each result becomes an `LLMTestCase`. This is the standard unit DeepEval metrics operate on.


### 🧪 Cell 8 — Package each result into a DeepEval `LLMTestCase`

An `LLMTestCase` is DeepEval's standard unit — one bundle containing the question (`input`), what the model said (`actual_output`), the human-verified correct answer (`expected_output`), and the chunks it saw (`retrieval_context`). Every metric below reads from this same bundle; building it once here means every metric downstream can just reuse it.

In [9]:
# Convert each RAG pipeline result into a DeepEval LLMTestCase.
#
# Every test case contains:
# - User input (query)
# - Model-generated answer
# - Expected (reference) answer
# - Retrieval context supplied to the LLM
#
# These test cases will be used by DeepEval metrics to evaluate
# both retrieval quality and answer quality.
test_cases = [
    LLMTestCase(
        input=result["query"],
        actual_output=result["actual_output"],
        expected_output=result["expected_output"],
        retrieval_context=result["retrieval_context"],
    )
    for result in results
]

# =============================================================================
# VERIFY THE GENERATED TEST CASES
# =============================================================================

# Display the total number of evaluation test cases created.
print(f"Built {len(test_cases)} LLMTestCase objects.")

# Inspect the contents of the first test case
# to ensure everything has been populated correctly.
print("\nExample test case fields:")

print("input:")
print(test_cases[0].input)

print("\nactual_output:")
print(test_cases[0].actual_output[:120], "...")

print("\nretrieval_context:")
print(test_cases[0].retrieval_context)

Built 6 LLMTestCase objects.

Example test case fields:
input:
What is the difference between LangGraph and a LangChain LCEL chain?

actual_output:
The difference between LangGraph and a LangChain LCEL chain is that LangGraph supports cycles and conditional branching, ...

retrieval_context:
['Unlike LCEL chains, which are directed acyclic graphs, LangGraph supports cycles and conditional branching.', 'LangGraph is a library for building stateful, multi-actor applications with LLMs, built on top of LangChain.', 'A StateGraph in LangGraph defines nodes, edges, and a shared state object that flows through the graph.']


---
## 6. Core RAG Metrics

All metrics below use an LLM internally (defaulting to `gpt-4o-mini` unless configured otherwise) to judge the relationship between `input`, `actual_output`, `retrieval_context`, and `expected_output`. Each metric produces a `0-1` score, a pass/fail against `threshold`, and a natural-language `reason`.

### 6.1 Generation Metrics

- **`FaithfulnessMetric`** - checks whether claims in `actual_output` are supported by `retrieval_context`. Catches hallucination *relative to the retrieved context* - the most important metric for RAG-specific hallucination.
- **`AnswerRelevancyMetric`** - checks whether `actual_output` actually addresses `input`, independent of whether it's grounded in context. Catches "correct-sounding but off-topic" answers.

### 6.2 Retrieval Metrics

- **`ContextualPrecisionMetric`** - of the chunks in `retrieval_context`, are the relevant ones ranked near the top? Penalizes retrieving relevant docs but burying them under irrelevant ones (compares against `expected_output` to judge relevance).
- **`ContextualRecallMetric`** - does `retrieval_context` contain everything needed to produce `expected_output`? Catches retrieval that's too narrow.
- **`ContextualRelevancyMetric`** - how much of `retrieval_context`, overall, is relevant to `input`? Catches retrieval that's too broad/noisy.

### 6.3 Hallucination (context-independent)

- **`HallucinationMetric`** - checks `actual_output` against a provided reference `context` for factual contradiction. Distinct from `FaithfulnessMetric` in that it's typically used to check faithfulness to a *fixed ground-truth document*, not necessarily what was retrieved.


### 📐 Cell 9 — Define the five core RAG metrics

Each line creates one grading rubric with a passing `threshold` of 0.7: **Faithfulness** (does the answer stick to what was retrieved — catches hallucination), **Answer Relevancy** (does it address the actual question, independent of grounding), and three retrieval-focused ones — **Contextual Precision/Recall/Relevancy** — checking whether the *retrieved chunks themselves* were good, not just the final answer. This is the same generation-vs-retrieval split established earlier: some of these grade what was found, others grade what was said.

In [10]:
# Faithfulness Metric
#
# Measures whether the generated answer is fully supported
# by the retrieved context.
#
# A low score indicates that the model may be hallucinating
# or introducing information that was not present in the
# retrieved documents.
faithfulness = FaithfulnessMetric(
    threshold=0.7
)

# Answer Relevancy Metric
#
# Evaluates how well the generated answer addresses
# the user's question.
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.7
)

# Contextual Precision Metric
#
# Measures whether the retrieved documents are relevant
# to the user's query while minimizing unnecessary context.
contextual_precision = ContextualPrecisionMetric(
    threshold=0.7
)

# Contextual Recall Metric
#
# Evaluates whether the retrieval system successfully
# retrieved all important information needed to answer
# the user's question.
contextual_recall = ContextualRecallMetric(
    threshold=0.7
)

# Contextual Relevancy Metric
#
# Measures the overall usefulness and relevance of the
# retrieved context for answering the user's query.
contextual_relevancy = ContextualRelevancyMetric(
    threshold=0.7
)

# GROUP ALL RAG METRICS

# Store all retrieval-related evaluation metrics in a list.
# These metrics will be passed to DeepEval during evaluation.
rag_metrics = [
    faithfulness,
    answer_relevancy,
    contextual_precision,
    contextual_recall,
    contextual_relevancy,
]

# Display the initialized metric names.
print(
    "RAG metrics initialized:",
    [metric.__class__.__name__ for metric in rag_metrics]
)

RAG metrics initialized: ['FaithfulnessMetric', 'AnswerRelevancyMetric', 'ContextualPrecisionMetric', 'ContextualRecallMetric', 'ContextualRelevancyMetric']


### 🏃 Cell 10 — Run all 5 metrics against all 6 test cases at once

`evaluate()` is DeepEval's batch runner — behind the scenes, for every test case it makes an LLM call per metric (the "LLM-as-judge" pattern), asking things like "is this answer supported by this context?", and prints a pass/fail scorecard with reasons.

In [11]:
# Run evaluation across all test cases and all RAG metrics
eval_results = evaluate(test_cases=test_cases, metrics=rag_metrics)

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-5.4, strict=False, async_mode=True)...

...

Task was destroyed but it is pending!
task: <Task pending name='Task-94' coro=<_async_in_context.<locals>.run_in_context() running at C:\Users\Avado\AppData\Roaming\Python\Python313\site-packages\ipykernel\utils.py:60> wait_for=<Task pending name='Task-95' coro=<Kernel.shell_main() running at C:\Users\Avado\AppData\Roaming\Python\Python313\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\Avado\AppData\Roaming\Python\Python313\site-packages\zmq\eventloop\zmqstream.py:564]>


Task was destroyed but it is pending!
task: <Task pending name='Task-97' coro=<Kernel.shell_main() running at C:\Users\Avado\AppData\Roaming\Python\Python313\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]>


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is the difference between LangGraph and a LangChain LCEL chain?                 │
│  │     Actual Output:      The difference between LangGraph and a LangChain LCEL chain is that LangGraph        │
│  │                         supports cycles and conditional branching, while LCEL chains are directed acyclic    │
│  │                         graphs. Additionally, LangGraph is designed for building stateful, multi-actor       │
│  │                         applications with LLMs and includes a StateGraph that defines nodes, edges, and a    │
│  │                         shared state object.                                                                 │
│  │     Expected Output:    LCEL chains are linear, acyclic pipelines, while LangGraph supports stateful         │
│  │                         graphs with cycles and conditional branching, making it suited for agentic           │
│  │                         workflows.                                                                           │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness         │ 1.00  │ 0.70      │ The score is 1.00 because there are no contradi...    │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.70      │ The score is 1.00 because the response appears ...    │
│        PASS  │ Contextual Precision │ 1.00  │ 0.70      │ The score is 1.00 because the retrieval context...    │
│        PASS  │ Contextual Recall    │ 1.00  │ 0.70      │ The score is 1.00 because sentence 1 is fully s...    │
│        FAIL  │ Contextual Relevancy │ 0.67  │ 0.70      │ The score is 0.67 because the retrieved context       │
│              │                      │       │           │ directly addresses the comparison by stating that     │
│              │                      │       │           │ "Unlike LCEL chains, which are directed acyclic       │
│              │                      │       │           │ graphs, LangGraph supports cycles and conditional     │
│              │                      │       │           │ branching" and that "LangGraph is a library for       │
│              │                      │       │           │ building stateful, multi-actor applications with      │
│              │                      │       │           │ LLMs," but it is only partially complete since        │
│              │                      │       │           │ there are no noted irrelevant statements and the      │
│              │                      │       │           │ context gives limited detail beyond "A StateGraph     │
│              │                      │       │           │ in LangGraph defines nodes, edges, and a shared       │
│              │                      │       │           │ state object."                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=327172;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 6.47s | token cost: 0.20446000000000003 USD)
» Test Results (6 total tests):
   » Pass Rate: 16.67% | Passed: 1 | Failed: 5

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### Inspecting a Single Metric in Detail

Running a metric directly (rather than through `evaluate()`) gives access to `.score` and `.reason` for a single test case - useful when debugging why a specific example scored low.


### 🔍 Cell 11 — Zoom into one specific test case's score and reasoning

Running `faithfulness.measure(sample_case)` directly (instead of through `evaluate()`) exposes `.score` (the 0-1 number) and `.reason` (the judge LLM's own explanation for that score) for just one question — useful for debugging *why* a particular answer scored low, rather than just seeing a pass/fail table.

In [12]:
# Select one evaluation example from the dataset.
#
# This sample is used to inspect individual metric scores
# before evaluating the entire dataset.
#
# Example:
# "How does hybrid search combine dense and sparse retrieval?"
sample_case = test_cases[1]

# FAITHFULNESS EVALUATION

# Measure whether the generated answer is fully grounded
# in the retrieved context.
#
# A high score indicates that the answer does not introduce
# unsupported or hallucinated information.
faithfulness.measure(sample_case)

print("Faithfulness score:", faithfulness.score)
print("Reason:", faithfulness.reason)

# CONTEXTUAL RECALL EVALUATION

# Measure whether the retrieved documents contain
# all the information necessary to answer the query.
#
# A high score indicates that the retrieval system
# successfully captured the important context.
contextual_recall.measure(sample_case)

print("\nContextual Recall score:", contextual_recall.score)
print("Reason:", contextual_recall.reason)

Output()

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

...

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered


Contextual Recall score: 0.0
Reason: The score is 0.00 because sentence 1 is not supported by the nodes in retrieval context: node 1 only says hybrid search combines dense semantic vectors with sparse lexical vectors, and node 2 identifies BM25 as sparse, but neither node states that dense and sparse retrieval are run separately, that two ranked lists are merged, or that fusion methods such as RRF are used.


---
## 7. LLM-as-a-Judge with `GEval`

The built-in metrics above cover faithfulness/relevancy/recall/precision well, but sometimes you need to score something more specific to your use case - tone, instructional clarity, conciseness, whether an answer correctly cites which topic it came from, etc. `GEval` lets you define **custom evaluation criteria in plain English**; DeepEval turns that into an LLM-judged rubric (using chain-of-thought scoring internally) without you having to write a custom metric class.

Below we define two custom judges relevant to this notebook's "teaching assistant" style RAG use case:

1. **Correctness** - does the answer align with the expected/reference answer factually (a more holistic, free-form correctness check than exact match)
2. **Instructional clarity** - is the answer written clearly and concisely enough to be useful in a teaching context, avoiding unnecessary hedging or verbosity


### ⚖️ Cell 12 — Build two custom judges with `GEval`, in plain English

The five metrics above cover generic RAG quality, but sometimes you care about something specific to *your* use case. `GEval` lets you write a grading rubric in a plain-English sentence — no custom code — and DeepEval turns it into an LLM-judged score. Here: a **Correctness** judge (does the answer match the reference facts, wording aside) and an **Instructional Clarity** judge (is it well-explained for a learner). This is the same LLM-as-judge idea from earlier, just aimed at a criterion *you* define instead of a built-in one.

In [13]:
# DEFINE CUSTOM LLM-AS-A-JUDGE METRICS

# DeepEval's GEval metric allows us to create custom evaluation
# criteria using an LLM as the evaluator.
#
# Unlike the built-in RAG metrics, GEval enables us to assess
# aspects such as correctness, clarity, style, completeness,
# or any custom quality dimension.

# CORRECTNESS METRIC

# Evaluate whether the generated answer is factually consistent
# with the expected (reference) answer.
#
# The wording does not need to be identical, but the generated
# response should:
# - Preserve the important facts.
# - Avoid contradictions.
# - Avoid omitting key information.
correctness_judge = GEval(
    name="Correctness",

    criteria=(
        "Determine whether the actual output is factually consistent "
        "with the expected output. The actual output does not need "
        "to match wording, but should not contradict or omit key "
        "facts present in the expected output."
    ),

    # Fields from the test case that will be supplied
    # to the evaluation model.
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],

    # Minimum passing score.
    threshold=0.7,
)

# INSTRUCTIONAL CLARITY METRIC

# Evaluate whether the generated answer is easy to understand
# for someone learning the topic.
#
# The answer should be:
# - Clear
# - Concise
# - Well-structured
# - Free from unnecessary repetition or filler
clarity_judge = GEval(
    name="Instructional Clarity",

    criteria=(
        "Evaluate whether the actual output is clear, concise, and "
        "appropriately explains the answer for someone learning the "
        "topic, without unnecessary hedging, filler, or repetition."
    ),

    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],

    threshold=0.7,
)

# GROUP CUSTOM EVALUATION METRICS

# Store all custom LLM-as-a-Judge metrics in a list.
judge_metrics = [
    correctness_judge,
    clarity_judge,
]

# Display the initialized metric names.
print(
    "LLM-as-judge metrics initialized:",
    [metric.name for metric in judge_metrics],
)

LLM-as-judge metrics initialized: ['Correctness', 'Instructional Clarity']


### 🏃 Cell 13 — Run the two custom judges across all test cases

Same `evaluate()` call as Cell 10, just pointed at the custom `judge_metrics` list instead of the built-in RAG metrics — same mechanism, different rubric.

In [14]:
judge_results = evaluate(test_cases=test_cases, metrics=judge_metrics)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Instructional Clarity [GEval] Metric! (using gpt-5.4, strict=False, 
async_mode=True)...

Output()

Task was destroyed but it is pending!
task: <Task pending name='Task-482' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
C:\Users\Avado\AppData\Roaming\Python\Python313\site-packages\ipykernel\utils.py:57> wait_for=<Task pending 
name='Task-483' coro=<Kernel.shell_main() running at 
C:\Users\Avado\AppData\Roaming\Python\Python313\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> 
cb=[ZMQStream._run_callback.<locals>._log_error() at 
C:\Users\Avado\AppData\Roaming\Python\Python313\site-packages\zmq\eventloop\zmqstream.py:564]>

C:\Python313\Lib\pathlib\_local.py:289: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  @property
RuntimeWarning: Enable tracemalloc to get the object allocation traceback

...

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How does hybrid search combine dense and sparse retrieval?                           │
│  │     Actual Output:      Hybrid search combines dense semantic vectors, which capture the meaning of the      │
│  │                         content, with sparse lexical vectors, which focus on specific terms and their        │
│  │                         occurrences, to improve retrieval accuracy.                                          │
│  │     Expected Output:    Hybrid search runs dense (semantic) and sparse (lexical/BM25-style) retrieval        │
│  │                         separately, then merges the two ranked lists using a fusion method such as           │
│  │                         Reciprocal Rank Fusion (RRF).                                                        │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                        ┃ Score ┃ Threshold ┃ Reason                                       │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Correctness [GEval]           │ 0.38  │ 0.70      │ The actual output correctly mentions         │
│              │                               │       │           │ combining dense semantic vectors with        │
│              │                               │       │           │ sparse lexical signals to improve            │
│              │                               │       │           │ retrieval, which aligns with the             │
│              │                               │       │           │ high-level idea in the expected output.      │
│              │                               │       │           │ However, it omits the key mechanism          │
│              │                               │       │           │ required by the expected answer: that        │
│              │                               │       │           │ hybrid search runs dense and sparse          │
│              │                               │       │           │ retrieval separately and then merges the     │
│              │                               │       │           │ two ranked lists with a fusion method such   │
│              │                               │       │           │ as Reciprocal Rank Fusion (RRF). Because     │
│              │                               │       │           │ the Input asks how hybrid search combines    │
│              │                               │       │           │ them, leaving out separate retrieval and     │
│              │                               │       │           │ rank fusion misses essential facts.          │
│        PASS  │ Instructional Clarity [GEval] │ 0.98  │ 0.70      │ The response directly answers how hybrid     │
│              │                               │       │   

⚠ WARNING: No hyperparameters logged.
» ]8;id=355620;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.34s | token cost: 0.05459 USD)
» Test Results (6 total tests):
   » Pass Rate: 83.33% | Passed: 5 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

`GEval` metrics can be defined for essentially any qualitative axis you care about - e.g., "penalizes answers that don't mention the topic tag," "penalizes overly technical jargon for a beginner audience," or "rewards answers that explicitly say when context is insufficient." This is the main lever for adapting evaluation to a specific product's definition of quality, beyond generic RAG grounding checks.


---
## 8. Aggregating Results into a Scorecard

`evaluate()` prints a results table, but for a notebook it's often more useful to pull scores into a simple DataFrame to compare across examples and metrics at a glance, and to spot systematic weak points (e.g., "contextual recall is consistently the lowest metric" → suggests widening `top_k` before rerank).


### 📊 Cell 14 — Combine every metric's score into one readable table

Instead of reading through printed pass/fail blocks one at a time, this re-runs every metric (5 built-in + 2 custom) against every test case and lays the results out as a Pandas table — one row per question, one column per metric. This is what makes it easy to spot a pattern like "Contextual Recall is consistently the weakest column," which would point you back at retrieval (widen `top_k`), not generation.

In [15]:
import pandas as pd

# Combine the built-in RAG metrics with the custom
# LLM-as-a-Judge metrics into a single evaluation suite.
all_metrics = rag_metrics + judge_metrics

# Store the evaluation scores for each test case.
rows = []

# Evaluate every test case across all metrics.
for test_case in test_cases:

    # Create a row for the final scorecard.
    row = {
        "query": test_case.input
    }

    # Compute each evaluation metric.
    for metric in all_metrics:

        # Run the metric on the current test case.
        metric.measure(test_case)

        # Use the metric's custom name if available;
        # otherwise use the class name.
        metric_name = getattr(
            metric,
            "name",
            metric.__class__.__name__,
        )

        # Store the score (rounded for readability).
        row[metric_name] = (
            round(metric.score, 3)
            if metric.score is not None
            else None
        )

    # Save the completed evaluation row.
    rows.append(row)

# BUILD THE EVALUATION SCORECARD

# Convert all evaluation results into a Pandas DataFrame.
#
# Each row represents one test case.
# Each column represents one evaluation metric.
scorecard = pd.DataFrame(rows)

# Display the complete evaluation scorecard.
scorecard

Output()

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

...

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002170A6DAF40> is already entered

,query,FaithfulnessMetric,AnswerRelevancyMetric,ContextualPrecisionMetric,ContextualRecallMetric,ContextualRelevancyMetric,Correctness,Instructional Clarity
0,What is the difference between LangGraph and a...,1.0,1.0,1.000,1.0,0.667,1.000,0.992
1,How does hybrid search combine dense and spars...,1.0,1.0,1.000,0.0,0.667,0.673,1.000
2,What open-source model can be used for reranki...,1.0,1.0,1.000,1.0,0.333,0.832,1.000
3,Why would I use RRF fusion instead of dense se...,1.0,1.0,0.583,0.0,1.000,0.710,0.894
4,What is Qdrant and what makes it different fro...,1.0,1.0,0.833,1.0,0.667,1.000,0.944
5,What programming language is Qdrant written in?,1.0,1.0,1.000,1.0,0.333,1.000,1.000


### 📉 Cell 15 — Average each metric down to one number

Collapses the whole scorecard into a single ranked list — which metric is systematically weakest across all 6 questions. This is the fastest way to answer "where should I focus improvement effort first?" without eyeballing every row individually.

In [16]:
# Average score per metric - quick view of systematic strengths/weaknesses
scorecard.drop(columns=["query"]).mean().sort_values().to_frame(name="avg_score")

,avg_score
ContextualRelevancyMetric,0.611167
ContextualRecallMetric,0.666667
Correctness,0.869167
ContextualPrecisionMetric,0.902667
Instructional Clarity,0.971667
AnswerRelevancyMetric,1.000000
FaithfulnessMetric,1.000000


### Choosing / Configuring the Judge Model

By default, DeepEval's metrics use `gpt-4o-mini` as the judge LLM. You can point metrics at a different OpenAI model (e.g., `gpt-4o` for a stricter/more capable judge) by passing `model=` to any metric constructor:

```python
FaithfulnessMetric(threshold=0.7, model="gpt-4o")
```

For fully open-source evaluation (no OpenAI dependency even for judging), DeepEval also supports plugging in local models via its `DeepEvalBaseLLM` interface - worth exploring if you need an entirely self-hosted eval pipeline, at the cost of judge quality/consistency compared to GPT-4-class models.


---
## 9. From Notebook to CI: Regression Testing

DeepEval is designed to run inside `pytest`, which means the same metrics used here can gate pull requests or catch regressions when you change the embedder, reranker, prompt, or fusion strategy.

A minimal `pytest` version of this notebook's evaluation might look like:

```python
# test_rag_pipeline.py
import pytest
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

@pytest.mark.parametrize("query,expected", EVAL_DATASET)
def test_rag_response(query, expected):
    out = graph.invoke({"query": query})
    test_case = LLMTestCase(
        input=query,
        actual_output=out["answer"],
        expected_output=expected,
        retrieval_context=[d["text"] for d in out["reranked_docs"]],
    )
    assert_test(test_case, [FaithfulnessMetric(threshold=0.7), AnswerRelevancyMetric(threshold=0.7)])
```

Run via `deepeval test run test_rag_pipeline.py` - this gives you pass/fail gating per metric per example, plus DeepEval's own reporting/dashboard integration if you use their hosted platform (Confident AI), which is optional.

**Practical tips for production eval sets:**
- Keep a **held-out** eval set that doesn't change when you tune prompts/retrieval - otherwise you're overfitting to your own test set
- Track metric trends over time (per model/prompt/retrieval version) rather than just pass/fail on a single run - a metric dropping from 0.91 to 0.78 average is a signal even if it's still "passing" threshold
- Pair `ContextualRecall`/`ContextualPrecision` trends with retrieval-only experiments (as in the main notebook's dense vs sparse vs hybrid comparison) to isolate whether regressions come from retrieval or generation
